# Séance 12 · Les agents, et le projet final · ⭐⭐⭐

**Niveau : ⭐⭐⭐ Avancé**

Dernière séance ! Jusqu'ici le modèle **parlait**. Aujourd'hui il **agit** : il décide d'appeler des outils Python que tu as écrits (chercher un Pokémon, calculer, fouiller dans tes notes), lit le résultat, et répond. C'est un **agent**, et on le construit sans aucun framework, en 30 lignes.

Ensuite, on finalise ton **portfolio GitHub** (4 projets) et on prépare la présentation devant le groupe et les invités.

Tout tourne dans **Google Colab** (menu *Exécution → Modifier le type d'exécution → T4 GPU*). Exécute chaque cellule avec `Maj + Entrée`.

**Livrable de la séance** : un portfolio GitHub complet et ta présentation finale.


## Préparation

La même cellule qu'aux séances 9 à 11 : elle prépare `llm(messages)`. Tout le notebook n'utilise que cette fonction.

In [ ]:
USE_MODEL = True   # ← mets False pour tester le notebook sans modèle (réponses factices, sans GPU)

import json, re
import ast, operator
import pandas as pd

# ---------- Mode démo : un faux LLM qui répond sans réseau ni GPU ----------
def llm_factice(messages):
    """Mode démo : un faux modèle qui suit le protocole OUTIL: nom(args) quand le prompt système le décrit."""
    systeme = " ".join(m["content"] for m in messages if m["role"] == "system")
    question = next(m["content"] for m in messages if m["role"] == "user")      # la question de départ
    ql = question.lower()
    if "OUTIL:" in systeme:                                                      # ---- mode agent ----
        resultats = [m["content"].split(":", 1)[1].split("\n")[0].strip()
                     for m in messages if m["role"] == "user" and m["content"].startswith("Résultat de l'outil")]
        if resultats:                                                            # on a déjà un résultat
            stat = re.search(r"vitesse|attaque|défense|pv", ql)
            facteur = re.search(r"(?:fois|multipli\w+ par|x|×)\s*(\d+)", ql)
            valeur = re.search((stat.group(0) if stat else "vitesse") + r"\D*(\d+)", resultats[-1].lower())
            if len(resultats) == 1 and facteur and valeur:                      # « la vitesse de X fois 2 » → calcul
                return f"OUTIL: calculer({valeur.group(1)} * {facteur.group(1)})"
            return "D'après mes outils : " + " ; ".join(resultats)
        expression = re.search(r"\d[\d\s+\-*/x×().]*\d", question)
        if expression and re.search(r"\d\s*[-+*/x×]\s*\d", question):
            return f"OUTIL: calculer({expression.group(0).replace('x', '*').replace('×', '*').strip()})"
        noms = re.findall(r"\b[A-Z][a-zé]{2,}\b", question[1:])                 # mots avec majuscule (sauf le 1er)
        if noms and re.search(r"pok[ée]mon|vitesse|attaque|défense|pv|type|rapide|fort", ql):
            return f"OUTIL: chercher_pokemon({noms[0]})"
        if re.search(r"notes|cours|séance|token|température|rag|api|rôle|agent|embedding|json", ql):
            return f"OUTIL: chercher_dans_mes_notes({question})"
        return "Je n'ai pas besoin d'outil : bonjour, je suis ton assistant !"
    if "questions" in ql and "invités" in ql:                                    # projet final
        return ("1. Qu'est-ce que ton programme fait exactement ? "
                "2. Qu'est-ce qui a été le plus difficile ? "
                "3. Est-ce que l'IA pourrait se tromper, et comment tu le saurais ?")
    if re.search(r"\d\s*[-+*/x×]\s*\d", ql):                                     # calcul sans outil : il se trompe
        return "Le résultat est 9 156."
    if re.search(r"pok[ée]mon|vitesse|attaque|pv", ql):                          # Pokémon sans outil : il invente
        return "Pikachu est un Pokémon Électrique avec 60 PV et une vitesse de 120."
    return "Bonne question ! Je ne suis pas certain, mais voici une réponse plausible : c'est un sujet intéressant."

# ---------- Le vrai modèle : petit modèle ouvert, gratuit, sans clé ----------

if USE_MODEL:
    %pip install -q transformers accelerate
    from transformers import pipeline
    _pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

def llm(messages, max_new_tokens=150, temperature=0.7):
    """Envoie une liste de messages au modèle et renvoie sa réponse (du texte)."""
    if not USE_MODEL:
        return llm_factice(messages)
    if temperature == 0:      # température 0 = toujours la réponse la plus probable
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
    else:
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature)
    return sortie[0]["generated_text"][-1]["content"].strip()

print("Modèle prêt :", "Qwen2.5-0.5B-Instruct" if USE_MODEL else "mode démo (llm_factice)")

**En option : le même appel avec une API.** Sur Colab, la clé est fournie par le formateur et rangée dans les **Secrets** (icône 🔑 à gauche), jamais dans le code. Décommente la cellule ci-dessous pour remplacer `llm()` par un gros modèle dans le cloud. Tout le reste du notebook ne change pas : il n'appelle que `llm(messages)`.

In [ ]:
# --- Variante API (à décommenter si le formateur a donné une clé) ---
# %pip install -q anthropic
# from google.colab import userdata          # les Secrets de Colab
# import anthropic
# client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
#
# def llm(messages, max_new_tokens=150, temperature=0.7):
#     systeme = " ".join(m["content"] for m in messages if m["role"] == "system")
#     autres = [m for m in messages if m["role"] != "system"]
#     rep = client.messages.create(model="claude-opus-5", max_tokens=max_new_tokens,
#                                  system=systeme or "Tu es un assistant sympa.", messages=autres)
#     return rep.content[0].text.strip()
#
# Même idée avec Mistral : pip install mistralai, clé dans userdata.get("MISTRAL_API_KEY"),
# client.chat.complete(model="mistral-small-latest", messages=messages).choices[0].message.content

## 1. Un agent, c'est un LLM qui agit

Rappel des séances 9 et 11 : un modèle **ne sait pas** ce qu'il n'a pas lu (les stats exactes d'un Pokémon, tes notes) et il **calcule mal**. Regarde.

In [ ]:
def demander(question, systeme="Tu es un assistant sympa qui répond en français, en une phrase."):
    return llm([{"role": "system", "content": systeme}, {"role": "user", "content": question}], temperature=0)

print("Pokémon :", demander("Quelle est la vitesse de Pikachu ?"))     # la vraie réponse : 90
print("Calcul  :", demander("Combien font 348 * 27 ?"))                 # la vraie réponse : 9 396

Un humain qui ne sait pas... **va chercher** : il ouvre le fichier, prend une calculatrice, relit son cours. Un **agent**, c'est exactement ça : un LLM à qui on donne des **outils** (des fonctions Python) et le droit de les appeler.

Analogie : le modèle est un stagiaire brillant mais sans accès à rien. Tu lui donnes un téléphone, une calculatrice et un classeur, et une règle : « quand tu as besoin de quelque chose, écris-moi *OUTIL: nom(arguments)* et je te rapporte le résultat ».

La boucle tient en une ligne :

`question → LLM → « OUTIL: chercher_pokemon(Pikachu) » → Python exécute → résultat → LLM → réponse`

C'est ce que font ChatGPT quand il « cherche sur le web », Claude Code quand il lit tes fichiers, ou un assistant qui réserve un billet : le même principe, avec plus d'outils et un plus gros modèle.

## 2. Trois outils écrits en Python

Un **outil**, c'est juste une fonction Python qui prend du texte et renvoie du texte. On en écrit trois :
1. `chercher_pokemon(nom)` : lit le fichier Pokémon de la séance 2.
2. `calculer(expression)` : une calculatrice **sécurisée** (on n'exécute jamais du code que le modèle a écrit tel quel !).
3. `chercher_dans_mes_notes(question)` : le RAG de la séance 11, version courte.

In [ ]:
# Outil 1 : chercher un Pokémon dans le fichier de la séance 2
URL_POKEMON = "https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv"
try:
    pokemon = pd.read_csv(URL_POKEMON)
except Exception as e:
    print("Pas de réseau ?", e, "→ on utilise 3 Pokémon de secours")
    pokemon = pd.DataFrame([
        {"Name": "Pikachu", "Type 1": "Electric", "HP": 35, "Attack": 55, "Defense": 40, "Speed": 90},
        {"Name": "Charizard", "Type 1": "Fire", "HP": 78, "Attack": 84, "Defense": 78, "Speed": 100},
        {"Name": "Snorlax", "Type 1": "Normal", "HP": 160, "Attack": 110, "Defense": 65, "Speed": 30},
    ])

def chercher_pokemon(nom):
    """Renvoie les stats d'un Pokémon (texte), ou un message s'il n'existe pas."""
    ligne = pokemon[pokemon["Name"].str.lower() == nom.strip().lower()]
    if ligne.empty:
        return f"Aucun Pokémon appelé {nom}."
    p = ligne.iloc[0]
    return f"{p['Name']} : type {p['Type 1']}, {p['HP']} PV, attaque {p['Attack']}, défense {p['Defense']}, vitesse {p['Speed']}."

print(chercher_pokemon("Pikachu"))
print(chercher_pokemon("Pikachou"))

Pour la calculatrice, surtout pas `eval(expression)` : si le modèle (ou quelqu'un qui lui parle) écrit `__import__("os").system("rm -rf /")`, `eval` l'exécuterait. On lit l'expression avec `ast` (l'arbre de syntaxe de Python) et on n'accepte **que** des nombres et les opérateurs `+ - * / ** %`.

In [ ]:
# Outil 2 : une calculatrice sécurisée (nombres et + - * / ** % uniquement)
OPERATIONS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
              ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod, ast.USub: operator.neg}

def _evaluer(noeud):
    if isinstance(noeud, ast.Constant) and isinstance(noeud.value, (int, float)):
        return noeud.value
    if isinstance(noeud, ast.UnaryOp) and type(noeud.op) in OPERATIONS:
        return OPERATIONS[type(noeud.op)](_evaluer(noeud.operand))
    if isinstance(noeud, ast.BinOp) and type(noeud.op) in OPERATIONS:
        gauche, droite = _evaluer(noeud.left), _evaluer(noeud.right)
        if isinstance(noeud.op, ast.Pow) and abs(droite) > 100:
            raise ValueError("exposant trop grand")
        return OPERATIONS[type(noeud.op)](gauche, droite)
    raise ValueError(f"élément interdit : {type(noeud).__name__}")

def calculer(expression):
    """Calcule une expression arithmétique. Refuse tout ce qui n'est pas un calcul."""
    try:
        arbre = ast.parse(expression.strip(), mode="eval")
        return f"{expression.strip()} = {_evaluer(arbre.body)}"
    except Exception as e:
        return f"Expression refusée ({e})."

print(calculer("348 * 27"))
print(calculer("(90 + 100) / 2"))
print(calculer('__import__("os").system("echo piraté")'))    # refusé !

In [ ]:
# Outil 3 : chercher dans mes notes (le RAG de la séance 11, version TF-IDF en 10 lignes)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

MES_NOTES = """Séance 9 : un token est un morceau de mot transformé en nombre. Le modèle ne voit pas les lettres, c'est pour ça qu'il compte mal les r de strawberry.

Séance 9 : un LLM fait une seule chose, prédire le token suivant, encore et encore. La température règle le hasard : 0 = toujours pareil, élevée = créatif puis délirant.

Séance 9 : un LLM hallucine, c'est-à-dire qu'il invente une réponse plausible quand il ne sait pas. Il a une date de connaissance et il calcule mal.

Séance 10 : une API, c'est comme un serveur de restaurant. On envoie une commande (la requête) et on reçoit un plat (la réponse), souvent en JSON.

Séance 10 : les trois rôles sont system (les consignes), user (l'utilisateur) et assistant (le modèle). Le prompt système donne la personnalité et les règles.

Séance 10 : un chatbot n'a pas de mémoire, il renvoie tout l'historique des messages à chaque tour. Pour réutiliser une réponse dans un programme, on demande du JSON.

Séance 11 : le RAG donne au modèle les bons passages de mes documents avant de poser la question. Les étapes sont découper, vectoriser, chercher, injecter, répondre.

Séance 11 : un embedding transforme un texte en vecteur. Des textes au sens proche donnent des points proches, et la similarité cosinus mesure cette proximité.

Séance 12 : un agent est un LLM qui peut agir. Il écrit OUTIL: nom(arguments), Python exécute l'outil et lui renvoie le résultat, puis il répond."""

notes_chunks = [p.strip() for p in MES_NOTES.split("\n\n") if p.strip()]     # découper
STOP_FR = "le la les l un une des du de d et ou à a au aux en dans sur par pour avec ce cet cette ces se son sa ses mon ma mes ton ta tes il elle on ne pas plus que qui quoi quel quelle quels est sont c'est".split()
tfidf = TfidfVectorizer(stop_words=STOP_FR).fit(notes_chunks)
notes_vecteurs = tfidf.transform(notes_chunks)                                # vectoriser

def chercher_dans_mes_notes(question, k=2):
    """Renvoie les k passages de mes notes les plus proches de la question."""
    scores = cosine_similarity(tfidf.transform([question]), notes_vecteurs)[0]   # chercher
    meilleurs = [i for i in scores.argsort()[::-1][:k] if scores[i] > 0.05]
    if not meilleurs:
        return "Rien trouvé dans les notes."
    return " | ".join(notes_chunks[i] for i in meilleurs)

print(chercher_dans_mes_notes("C'est quoi la température ?"))

**Exercice** : écris un 4e outil, `plus_rapide(type_pokemon)`, qui renvoie le nom et la vitesse du Pokémon le plus rapide d'un type (`"Fire"`, `"Water"`...). Teste-le avec 2 types. (Indice : filtre sur `Type 1`, puis `.sort_values("Speed")` ou `.idxmax()`.)

In [ ]:
# À toi
def plus_rapide(type_pokemon):
    du_type = pokemon[pokemon["Type 1"].str.lower() == type_pokemon.strip().lower()]
    if du_type.empty:
        return f"Aucun Pokémon de type {type_pokemon}."
    # ... trouve la ligne avec la plus grande vitesse et renvoie "Nom : vitesse N"
    return "à compléter"

print(plus_rapide("Fire"))

<details><summary>Solution</summary>

```python
def plus_rapide(type_pokemon):
    du_type = pokemon[pokemon["Type 1"].str.lower() == type_pokemon.strip().lower()]
    if du_type.empty:
        return f"Aucun Pokémon de type {type_pokemon}."
    p = du_type.loc[du_type["Speed"].idxmax()]
    return f"Le {type_pokemon} le plus rapide est {p['Name']} (vitesse {p['Speed']})."

print(plus_rapide("Fire"))
print(plus_rapide("Water"))
```

</details>

## 3. La boucle de l'agent (sans framework)

Le protocole est écrit dans le prompt système : « pour utiliser un outil, réponds `OUTIL: nom(arguments)` ». Notre boucle Python :
1. envoie la question au modèle ;
2. si sa réponse commence par `OUTIL:`, on repère le nom et les arguments avec une **expression régulière**, on exécute la fonction, et on renvoie le résultat au modèle (comme un nouveau message) ;
3. sinon, c'est la réponse finale.

On limite le nombre d'étapes pour que l'agent ne tourne pas en rond.

In [ ]:
OUTILS = {"chercher_pokemon": chercher_pokemon, "calculer": calculer, "chercher_dans_mes_notes": chercher_dans_mes_notes}

SYSTEME_AGENT = """Tu es un assistant qui peut utiliser des outils. Tes outils :
- chercher_pokemon(nom) : les statistiques d'un Pokémon (PV, attaque, défense, vitesse). Ex : OUTIL: chercher_pokemon(Pikachu)
- calculer(expression) : calcule une expression mathématique. Ex : OUTIL: calculer(348 * 27)
- chercher_dans_mes_notes(question) : cherche dans les notes de cours de l'élève. Ex : OUTIL: chercher_dans_mes_notes(c'est quoi un token)
Pour utiliser un outil, réponds EXACTEMENT une ligne de la forme OUTIL: nom(arguments), et rien d'autre.
Quand tu reçois le résultat de l'outil, réponds à la question en français, en une phrase, à partir de ce résultat.
Si tu n'as pas besoin d'outil, réponds directement en une phrase."""

def agent(question, max_etapes=4, afficher=True):
    """La boucle : le modèle parle → si OUTIL: on exécute et on lui renvoie le résultat → sinon c'est fini."""
    messages = [{"role": "system", "content": SYSTEME_AGENT}, {"role": "user", "content": question}]
    for etape in range(1, max_etapes + 1):
        reponse = llm(messages, max_new_tokens=60, temperature=0)
        appel = re.search(r"OUTIL\s*:\s*(\w+)\((.*)\)", reponse, re.DOTALL)
        if not appel:                                             # pas d'outil demandé : réponse finale
            return reponse
        nom, args = appel.group(1), appel.group(2).strip().strip("'\"")
        resultat = OUTILS[nom](args) if nom in OUTILS else f"Outil inconnu : {nom}. Outils : {list(OUTILS)}"
        if afficher:
            print(f"  [étape {etape}] modèle → {reponse.strip()}")
            print(f"  [étape {etape}] outil  → {resultat}")
        messages.append({"role": "assistant", "content": reponse})
        messages.append({"role": "user", "content": f"Résultat de l'outil : {resultat}\nRéponds maintenant à la question."})
    return f"Je n'ai pas réussi en {max_etapes} étapes."

print(agent("Quelle est la vitesse de Pikachu ?"))

In [ ]:
print(agent("Combien font 348 * 27 ?"))

**Exercice** : pose à l'agent une question sur un Pokémon qui n'existe pas (« Pikachou »), puis une question qui n'a besoin d'aucun outil (« Bonjour, qui es-tu ? »). Que fait-il dans chaque cas ? Ensuite, ajoute ton outil `plus_rapide` au dictionnaire `OUTILS` **et** à la liste dans `SYSTEME_AGENT` (le modèle ne connaît que ce qu'on lui décrit !).

In [ ]:
# À toi
print(agent("Quelle est la vitesse de Pikachou ?"))
print()
print(agent("Bonjour, qui es-tu ?"))

<details><summary>Solution</summary>

```python
# Pokémon inconnu : l'outil répond "Aucun Pokémon appelé Pikachou", et le modèle le dit (un gros modèle
# propose parfois une correction : "tu voulais dire Pikachu ?"). Sans besoin d'outil : il répond directement.
OUTILS["plus_rapide"] = plus_rapide
SYSTEME_AGENT = SYSTEME_AGENT.replace(
    "Pour utiliser un outil",
    "- plus_rapide(type) : le Pokémon le plus rapide d'un type. Ex : OUTIL: plus_rapide(Fire)\nPour utiliser un outil")
print(agent("Quel est le Pokémon de type Fire le plus rapide ?"))
```

</details>

## 4. Démo : l'agent décide d'aller chercher dans le RAG

Maintenant une question sur **tes notes**. Le modèle n'a pas la réponse, il n'a pas de Pokémon à chercher, rien à calculer : il doit **choisir** le bon outil tout seul. C'est ça, la différence entre un chatbot et un agent : il décide de l'étape suivante.

In [ ]:
print(agent("D'après mes notes de cours, quelles sont les étapes du RAG ?"))
print()
print(agent("Dans mes notes, à quoi sert la température ?"))

Et un enchaînement : deux outils à la suite. Le modèle doit d'abord chercher la vitesse, **puis** appeler la calculatrice avec le résultat.

In [ ]:
print(agent("Combien fait la vitesse de Pikachu fois 3 ?"))

**Exercice** : ajoute un paragraphe à `MES_NOTES` sur un sujet de ton choix (ta matière préférée, un jeu...), reconstruis l'index (les 3 lignes `notes_chunks` / `tfidf` / `notes_vecteurs`), puis pose une question dessus à l'agent. Choisit-il le bon outil ?

In [ ]:
# À toi
MES_NOTES += "\n\nCours de SVT : la photosynthèse transforme la lumière, l'eau et le CO2 en sucre et en oxygène, dans les feuilles."
notes_chunks = [p.strip() for p in MES_NOTES.split("\n\n") if p.strip()]
tfidf = TfidfVectorizer(stop_words=STOP_FR).fit(notes_chunks)
notes_vecteurs = tfidf.transform(notes_chunks)

print(agent("Dans mes notes de cours, c'est quoi la photosynthèse ?"))

<details><summary>Solution</summary>

```python
# Si l'agent répond sans outil (il "sait" ce qu'est la photosynthèse), c'est normal : le prompt dit
# "si tu n'as pas besoin d'outil, réponds directement". Pour le forcer à citer TES notes, ajoute dans
# SYSTEME_AGENT : "Pour toute question qui mentionne les notes ou le cours, utilise chercher_dans_mes_notes."
```

</details>

## 5. Ce qui peut rater

Avec le petit modèle de Colab (0,5 milliard de paramètres), tu vas voir des ratés :
- il **oublie le format** (« Je vais chercher Pikachu ! » au lieu de `OUTIL: chercher_pokemon(Pikachu)`) ;
- il invente un outil qui n'existe pas, ou met les mauvais arguments ;
- il **n'appelle pas l'outil** et répond de tête (donc il hallucine) ;
- il tourne en rond (d'où `max_etapes`).

Les gros modèles font beaucoup mieux : ils ont été entraînés exprès à appeler des outils (on parle de *function calling*). Mais même eux se trompent : un agent sérieux **vérifie** les arguments, **limite** ce que chaque outil peut faire, et **garde un humain** dans la boucle pour les actions importantes (envoyer un mail, payer, effacer).

Le banc de test ci-dessous mesure ton agent : sur chaque question, a-t-il utilisé le bon outil ?

In [ ]:
banc_de_test = [
    ("Quelle est la défense de Charizard ?",                 "chercher_pokemon"),
    ("Combien font 12 * 12 + 1 ?",                            "calculer"),
    ("Dans mes notes de cours, c'est quoi un token ?",       "chercher_dans_mes_notes"),
    ("Quels sont les PV de Snorlax ?",                        "chercher_pokemon"),
    ("Que disent mes notes de cours sur les trois rôles ?",  "chercher_dans_mes_notes"),
    ("Combien font 1000 / 8 ?",                               "calculer"),
]

def outil_utilise(question):
    """Renvoie le nom du premier outil que le modèle demande (ou None)."""
    reponse = llm([{"role": "system", "content": SYSTEME_AGENT}, {"role": "user", "content": question}], max_new_tokens=60, temperature=0)
    appel = re.search(r"OUTIL\s*:\s*(\w+)\(", reponse)
    return appel.group(1) if appel else None

reussites = 0
for question, attendu in banc_de_test:
    obtenu = outil_utilise(question)
    ok = obtenu == attendu
    reussites += ok
    print(f"{'✓' if ok else '✗'} {question[:45]:45} attendu {attendu:24} obtenu {obtenu}")
print(f"\nScore de l'agent : {reussites} / {len(banc_de_test)}")

**Exercice** : rends la boucle plus tolérante. Modifie l'expression régulière de `agent` pour accepter aussi `outil : calculer(2+2)` (minuscules, espace avant les deux-points) et une phrase autour (« Je vais utiliser OUTIL: calculer(2+2) pour ça »). Indice : `re.IGNORECASE`, et `re.search` cherche déjà n'importe où dans le texte.

In [ ]:
# À toi
def extraire_appel(reponse):
    appel = re.search(r"OUTIL\s*:\s*(\w+)\((.*)\)", reponse, re.DOTALL)
    return (appel.group(1), appel.group(2).strip()) if appel else None

for essai in ["OUTIL: calculer(2+2)", "outil : calculer(2+2)", "Je vais utiliser OUTIL: calculer(2+2) pour ça."]:
    print(f"{essai!r:55} → {extraire_appel(essai)}")

<details><summary>Solution</summary>

```python
def extraire_appel(reponse):
    appel = re.search(r"OUTIL\s*:\s*(\w+)\((.*?)\)", reponse, re.DOTALL | re.IGNORECASE)
    return (appel.group(1), appel.group(2).strip()) if appel else None

for essai in ["OUTIL: calculer(2+2)", "outil : calculer(2+2)", "Je vais utiliser OUTIL: calculer(2+2) pour ça."]:
    print(f"{essai!r:55} → {extraire_appel(essai)}")
# Le (.*?) "paresseux" s'arrête à la première parenthèse fermante : plus de "pour ça." aspiré dans les arguments.
```

</details>

## 6. Les bonnes pratiques avec l'IA

Tu sais maintenant comment ça marche sous le capot. Voilà ce que ça change dans ta façon de l'utiliser :

- **Vérifier les réponses.** Un LLM prédit des mots plausibles, pas des vérités (séance 9). Pour une date, un chiffre, une citation, un médicament : une source fiable en plus. Plus la réponse est précise et importante, plus tu vérifies.
- **Ne pas partager de données personnelles.** Ton adresse, tes mots de passe, une photo d'un ami, les notes de quelqu'un d'autre : ce que tu tapes peut être stocké et relu. Règle simple : si tu ne l'écrirais pas sur un mur public, tu ne l'envoies pas à un chatbot.
- **Savoir quand l'IA se trompe.** Signaux d'alerte : elle est très sûre d'elle sur un sujet rare, elle donne une source que tu ne trouves pas, elle calcule, elle parle d'après sa date de connaissance, elle répond à une question hors de ses documents (séance 11). Demande-lui d'expliquer son raisonnement, pose la même question autrement.
- **Apprendre plutôt que copier.** Copier une réponse, c'est comme regarder quelqu'un faire des pompes : ça ne muscle rien. Utilise l'IA pour te faire **expliquer**, te faire **interroger** (ton coach de la séance 10), relire ton code et te dire *pourquoi* ça marche. Le test : peux-tu le refaire sans elle ?
- **Rester aux commandes.** Un agent qui agit à ta place (envoyer, acheter, effacer) doit te demander avant les actions importantes. C'est toi qui décides.

Petit quiz pour vérifier : réponds `True` (bonne idée) ou `False` (mauvaise idée) pour chaque situation.

In [ ]:
quiz = [
    "Je demande à un chatbot la date de naissance d'un scientifique peu connu et je la mets dans mon exposé sans vérifier.",
    "Je colle mon code qui plante et je demande : explique-moi pourquoi ça plante, sans me donner la correction.",
    "Je donne au chatbot le prénom, le nom et la classe d'un camarade pour qu'il écrive une blague sur lui.",
    "Pour un calcul avec beaucoup de chiffres, je fais faire le calcul par Python (ou un outil), pas par le modèle.",
    "Le modèle répond avec assurance à une question sur mon jeu inventé de la séance 11, donc c'est sûrement vrai.",
    "Je demande au chatbot de me poser 5 questions sur mon cours d'histoire et je réponds sans regarder mes notes.",
]
REPONSES = [False, True, False, True, False, True]

# Question : remplis ta liste (True = bonne idée, False = mauvaise idée), puis lance la cellule
mes_reponses = [None, None, None, None, None, None]

score = 0
for i, (situation, attendu, donne) in enumerate(zip(quiz, REPONSES, mes_reponses), 1):
    if donne is None:
        print(f"? {i}. {situation[:80]}... → à toi de répondre")
        continue
    ok = donne == attendu
    score += ok
    print(f"{'✓' if ok else '✗'} {i}. {situation[:80]}... → réponse attendue : {attendu}")
print(f"\nScore : {score} / {len(quiz)}")

## 7. Projet final : ton portfolio et ta présentation (80 min)

Un **portfolio**, c'est ta vitrine : un dépôt GitHub (séance 3) qui montre ce que tu sais faire. Quelqu'un qui l'ouvre doit comprendre en 2 minutes qui tu es et ce que tu as construit. Voici les 4 projets attendus, un par bloc :

| # | Projet | Séances | Ce qu'on doit y trouver |
|---|---|---|---|
| 1 | Analyse d'un dataset de ton choix | 2, 4, 5 | le notebook, 3 graphiques, 3 phrases de conclusion |
| 2 | Dashboard et premier modèle | 5, 6 | le dashboard (ou ses captures), le modèle et son score |
| 3 | Compétition Kaggle Titanic | 7, 8 | le notebook, ton score et ta place au classement |
| 4 | Assistant IA (chatbot, RAG ou agent) | 10, 11, 12 | le notebook, le prompt système, un exemple de conversation |

Étapes : (1) coche la check-list et corrige ce qui manque, (2) génère le README du portfolio, (3) remplis ta fiche projet pour la présentation, (4) répète une fois avec un camarade.

In [ ]:
# Question 1 : la check-list. Mets True quand c'est fait, puis lance la cellule pour voir ta progression
portfolio = {
    "1. Analyse d'un dataset":      {"notebook sur GitHub": True,  "3 graphiques lisibles": True,  "3 phrases de conclusion": False, "README de 5 lignes": False},
    "2. Dashboard et modèle":       {"notebook sur GitHub": False, "capture du dashboard": False,  "score du modèle indiqué": False, "README de 5 lignes": False},
    "3. Kaggle Titanic":            {"notebook sur GitHub": False, "score Kaggle indiqué": False,  "ce que j'ai essayé (3 lignes)": False, "README de 5 lignes": False},
    "4. Assistant IA":              {"notebook sur GitHub": False, "prompt système visible": False, "exemple de conversation": False, "README de 5 lignes": False},
    "Le dépôt lui-même":            {"README d'accueil (qui je suis, les 4 projets)": False, "pas de clé d'API ni de données perso": True, "lien du dépôt testé dans un navigateur privé": False},
}

total = fait = 0
for projet, cases in portfolio.items():
    n, ok = len(cases), sum(cases.values())
    total, fait = total + n, fait + ok
    barre = "█" * ok + "░" * (n - ok)
    print(f"{barre}  {projet:28} {ok}/{n}", "" if ok == n else "→ manque : " + ", ".join(k for k, v in cases.items() if not v))
print(f"\nPortfolio : {fait}/{total} cases ({100 * fait // total} %)")

In [ ]:
# Question 2 : génère le README d'accueil de ton portfolio (copie le résultat dans README.md sur GitHub)
PRENOM = "Nova"
UNE_PHRASE_SUR_MOI = "Je débute, j'aime les jeux vidéo et j'ai découvert la data et l'IA pendant cet atelier."
PROJET_PREFERE = "4. Assistant IA"

readme = f"""# Portfolio de {PRENOM}

{UNE_PHRASE_SUR_MOI}

## Mes 4 projets

| Projet | Ce que j'ai fait | Lien |
|---|---|---|
"""
for projet in list(portfolio)[:4]:
    etoile = " ⭐" if projet == PROJET_PREFERE else ""
    fichier = projet.split(". ")[1].lower().replace(" ", "_").replace("'", "") + ".ipynb"
    readme += f"| {projet}{etoile} | (1 phrase) | [notebook](./{fichier}) |\n"
readme += """
## Ce que j'ai appris

- (Python et pandas) ...
- (machine learning) ...
- (IA générative) ...

Atelier « Python, Data Science et IA générative », 12 séances. Tout tourne dans Google Colab, gratuitement.
"""
print(readme)

### Le gabarit de présentation (3 minutes, devant le groupe et les invités)

Les invités ne connaissent ni pandas ni les tokens. Ton but : qu'ils comprennent **ce que tu as fait** et **pourquoi c'est cool**, pas comment marche TF-IDF.

| Diapo | Durée | Contenu |
|---|---|---|
| 1. Le projet en une phrase | 20 s | « J'ai construit ... qui permet de ... » |
| 2. Le problème | 30 s | pourquoi c'était intéressant ou difficile, avec un exemple concret |
| 3. La démo | 1 min | **montre-le en vrai** : une question à ton assistant, un graphique, ton score Kaggle |
| 4. Ce que j'ai appris | 40 s | 2 choses que tu ne savais pas faire avant, 1 fois où ça a raté et ce que tu as compris |
| 5. La suite | 30 s | ce que tu ferais avec plus de temps, et où trouver ton portfolio (le lien !) |

Conseils : un seul mot technique par diapo, expliqué en une phrase. Parle de ce qui a raté, c'est ce qui rend l'histoire vraie. Et prévois une réponse à : « et si l'IA se trompe ? ».

In [ ]:
# Question 3 : ma fiche projet (remplis chaque champ, la cellule vérifie qu'il ne manque rien)
ma_fiche = {
    "Prénom":                      "Nova",
    "Projet présenté":             "Un assistant qui répond aux questions sur mes notes de cours (RAG + agent)",
    "En une phrase pour mamie":    "C'est un programme à qui je donne mes cours et qui répond à mes questions dessus.",
    "Le problème":                 "Un chatbot normal invente quand il ne connaît pas mes cours.",
    "Ma démo (ce que je montre)":  "Je pose 2 questions : une où il répond juste, une hors sujet où il dit qu'il ne sait pas.",
    "Ce que j'ai appris (2)":      "",
    "Une fois où ça a raté":       "",
    "La suite":                    "",
    "Lien de mon portfolio":       "",
}

print("=== MA FICHE PROJET ===")
for champ, valeur in ma_fiche.items():
    print(f"{champ:28} : {valeur if valeur else '⚠️  à remplir'}")
manquants = [c for c, v in ma_fiche.items() if not v]
print(f"\n{'Fiche complète, prêt pour la présentation !' if not manquants else str(len(manquants)) + ' champ(s) à remplir'}")

In [ ]:
# Question 4 : prépare les questions du public. Demande au modèle 3 questions que des invités pourraient poser,
# puis écris tes réponses dans `mes_reponses_au_public`
questions_public = demander(
    f"Voici un projet présenté par un débutant : {ma_fiche['Projet présenté']}. "
    "Écris 3 questions simples que des invités pourraient poser après la présentation, numérotées.")
print(questions_public)

mes_reponses_au_public = {
    "Question 1": "",
    "Question 2": "",
    "Question 3": "",
}

**Pour aller plus loin** (après l'atelier) : (a) donne un vrai outil web à ton agent (`requests.get` sur l'API météo de la séance 4) ; (b) branche le RAG de la séance 11 avec `sentence-transformers` à la place de TF-IDF ; (c) mets ton assistant dans une page Streamlit (bonus de la séance 10) et ajoute le lien dans ton portfolio.

## À retenir

- Un **agent** = un LLM + des **outils** (des fonctions Python) + une **boucle** : le modèle demande `OUTIL: nom(args)`, Python exécute, le résultat repart au modèle, jusqu'à la réponse finale.
- Le modèle ne connaît que les outils qu'on lui **décrit** dans le prompt système. Un outil = une fonction qui prend du texte et renvoie du texte.
- Jamais d'`eval` sur ce qu'écrit un modèle : chaque outil **vérifie** ses arguments et ne fait qu'une chose.
- Un petit modèle oublie le format ou n'appelle pas l'outil ; les gros sont entraînés pour ça, mais on limite quand même les étapes et on garde un humain pour les actions importantes.
- Bonnes pratiques : **vérifier**, ne pas donner de **données perso**, repérer **quand l'IA se trompe**, s'en servir pour **apprendre** et pas pour copier.
- Ton **portfolio GitHub** raconte 12 séances en 4 projets : c'est lui que tu montres, pas ton diplôme de l'atelier.

## Pour montrer aux autres

C'est la présentation finale (3 minutes chacun, devant le groupe et les invités). Suis ta fiche projet et réponds à :
1. Qu'est-ce que tu as construit, en une phrase que ta grand-mère comprend ? Montre-le en vrai.
2. Une fois où ça a raté (le modèle a inventé, le score était mauvais, le code plantait) : qu'as-tu compris ?
3. Qu'est-ce que tu ferais avec 3 séances de plus ?

Liens gratuits
- Ton portfolio : https://github.com (et le guide de démarrage en français : https://docs.github.com/fr/get-started)
- Cours gratuit sur les agents, par Hugging Face : https://huggingface.co/learn/agents-course
- Continuer les compétitions : https://www.kaggle.com/competitions
- Google Colab, pour tout refaire chez toi : https://colab.research.google.com